# 08 LightGBM Optuna Tuning

This notebook tunes LightGBM with Optuna on the registry-driven M1, M2, M3, and M4 feature sets from notebook 06. Notebook 07 provides model-family screening; this notebook selects validation-period LightGBM configurations for later rolling-origin evaluation.

M3 remains a non-operational realised-weather point-information benchmark because realised target-day weather is not available at forecast origin. M1, M2, and M4 must not contain leakage-risk columns. Tuning uses a time-safe train/validation split that lies entirely before the reserved rolling-origin evaluation period, and this notebook does not score final evaluation rows.


## Configuration

`RUN_MODE` controls the Optuna trial budget and horizon subset. `RUN_MODE = "quick"` uses `N_TRIALS = 10` and horizons `[0, 1, 2, 3, 7, 10]` for development checks. `RUN_MODE = "full"` uses `N_TRIALS = 50` and all horizons `0..10` for thesis-level tuning. Output files use `_quick` or `_full` suffixes so quick runs do not overwrite full-run outputs.


In [ ]:
RUN_MODE = "full"  # change to "full" for the thesis-level tuning run.
assert RUN_MODE in {"quick", "full"}, f"Unknown RUN_MODE: {RUN_MODE!r}"

if RUN_MODE == "quick":
    N_TRIALS = 10
    HORIZONS_TO_TUNE = [0, 1, 2, 3, 7, 10]
else:
    N_TRIALS = 50
    HORIZONS_TO_TUNE = list(range(0, 11))

# Tune all four registry feature sets;
# M3 remains non-operational because it uses realised target-day weather.
TUNING_FEATURE_SETS = ["M1", "M2", "M3", "M4"]

RANDOM_STATE = 42
N_JOBS = -1  # use all available CPU cores.
CLIP_NEGATIVE_PREDICTIONS = True

OPTUNA_SAMPLER_SEED = RANDOM_STATE
OPTUNA_DIRECTION = "minimize"

# Median pruning is enabled by default in full mode.
ENABLE_OPTUNA_PRUNING = (RUN_MODE == "full")


## Inputs and leakage boundary

Inputs are `data/processed/ml_forecast_panel_full.parquet` and `outputs/ml_panel/ml_panel_feature_registry.csv`. The tuning train period is `target_date <= 2024-03-31`; the tuning validation period is `2024-04-01 <= target_date <= 2024-07-31`. Rows reserved for final rolling-origin evaluation, defined by `origin_date >= 2024-08-01` and `target_date <= 2025-07-31`, are not used here.

The validation split is also origin-aware: validation rows must have `origin_date <= 2024-07-31`. This prevents forecasts whose origin lies in the reserved rolling-origin period from entering tuning.


In [ ]:
import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project "
        "folder or make sure README_FOR_CODEX.md, AGENTS.md, or pyproject.toml exists."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_ML_PANEL_DIR = OUTPUT_DIR / "ml_panel"
OUTPUT_TUNING_DIR = OUTPUT_DIR / "lightgbm_tuning"

ML_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
FEATURE_REGISTRY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_feature_registry.csv"

OUTPUT_TUNING_DIR.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


def require_file(path, description):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {description}: {project_relative(path)}. "
            "Rerun the producing notebook before running this notebook."
        )


require_file(FEATURE_REGISTRY_PATH, "feature registry from notebook 06")
require_file(ML_PANEL_PATH, "ML forecast panel from notebook 06")

# Time-safe split; reserved final-evaluation rows are not used for fitting or scoring.
TUNING_TRAIN_END = pd.Timestamp("2024-03-31")
TUNING_VAL_START = pd.Timestamp("2024-04-01")
TUNING_VAL_END = pd.Timestamp("2024-07-31")

# Recorded only as forward references for the rolling-origin notebook.
RESERVED_ROLLING_ORIGIN_START = pd.Timestamp("2024-08-01")
RESERVED_ROLLING_ORIGIN_END   = pd.Timestamp("2025-07-31")

def tuning_path(stem: str, ext: str) -> Path:
    return OUTPUT_TUNING_DIR / f"{stem}_{RUN_MODE}.{ext}"

BEST_PARAMS_CSV   = tuning_path("lightgbm_optuna_best_params",         "csv")
BEST_PARAMS_JSON  = tuning_path("lightgbm_optuna_best_params",         "json")
TRIALS_CSV        = tuning_path("lightgbm_optuna_trials",              "csv")
VAL_METRICS_CSV   = tuning_path("lightgbm_optuna_validation_metrics",  "csv")
FEATURE_SET_CSV   = tuning_path("lightgbm_optuna_feature_set_summary", "csv")
RUNTIME_CSV       = tuning_path("lightgbm_optuna_runtime_summary",     "csv")
FIG_DIR           = OUTPUT_TUNING_DIR / f"figures_{RUN_MODE}"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"ML panel input:        {project_relative(ML_PANEL_PATH)}")
print(f"Feature registry:      {project_relative(FEATURE_REGISTRY_PATH)}")
print(f"Tuning outputs folder: {project_relative(OUTPUT_TUNING_DIR)}")
print()
print(f"RUN_MODE:        {RUN_MODE}")
print(f"N_TRIALS:        {N_TRIALS}")
print(f"HORIZONS:        {HORIZONS_TO_TUNE}")
print(f"TUNE FEATURE SETS: {TUNING_FEATURE_SETS}")
print(f"Tuning train:    target_date <= {TUNING_TRAIN_END.date()}")
print(f"Tuning val:      {TUNING_VAL_START.date()} <= target_date <= {TUNING_VAL_END.date()}")
print(f"Reserved (not touched): origin_date >= {RESERVED_ROLLING_ORIGIN_START.date()} "
      f"AND target_date <= {RESERVED_ROLLING_ORIGIN_END.date()}")


## Imports and package availability

LightGBM and Optuna are required. The notebook fails early with a clear message if either package is unavailable.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor
    _LIGHTGBM_OK = True
except Exception as exc:  # pragma: no cover
    _LIGHTGBM_OK = False
    raise ImportError(
        f"LightGBM is not available: {type(exc).__name__}: {exc}. "
        "Install lightgbm in the csvi_env conda environment before running notebook 08."
    )

try:
    import optuna  # noqa: F401
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner, NopPruner
    _OPTUNA_OK = True
except Exception as exc:  # pragma: no cover
    _OPTUNA_OK = False
    raise ImportError(
        f"Optuna is not available: {type(exc).__name__}: {exc}. "
        "Install optuna in the csvi_env conda environment before running notebook 08."
    )

# Suppress Optuna per-trial logging unless warnings/errors occur.
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Matplotlib plots are optional diagnostics.
try:
    import matplotlib.pyplot as plt
    _MPL_OK = True
except Exception:
    _MPL_OK = False

print(f"lightgbm available: {_LIGHTGBM_OK}")
print(f"optuna available:   {_OPTUNA_OK}")
print(f"matplotlib (optional): {_MPL_OK}")


## Feature registry loading

Feature sets are reconstructed from `outputs/ml_panel/ml_panel_feature_registry.csv` using the same registry logic as notebook 07. The four flags `allowed_m1_baseline`, `allowed_m2_synthetic_point_weather`, `allowed_m3_perfect_information`, and `allowed_m4_synthetic_engineered_weather` define column eligibility. Key columns, target columns, and `origin_season` are excluded from the feature lists, while `AvdelingID` and `Analyse_Kategori` are added back as categorical identifiers.

Leakage assertions run before tuning: M1, M2, and M4 must contain no `leakage_risk == True` columns; M3 may contain only realised-weather perfect-information columns; and no feature set may contain `target_date`, `origin_date`, `horizon`, or target columns.


In [ ]:
feature_registry = pd.read_csv(FEATURE_REGISTRY_PATH)


def _to_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


for column in [
    "allowed_m1_baseline",
    "allowed_m2_synthetic_point_weather",
    "allowed_m3_perfect_information",
    "allowed_m4_synthetic_engineered_weather",
    "leakage_risk",
    "known_at_origin",
]:
    feature_registry[column] = _to_bool(feature_registry[column])

KEY_COLUMNS = feature_registry.loc[feature_registry["role"] == "key", "column"].tolist()
TARGET_COLUMNS = feature_registry.loc[
    feature_registry["role"].isin(["target", "robustness_target"]), "column"
].tolist()

CATEGORICAL_FEATURE_CANDIDATES = [
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "HelligdagNavn",
    "season",
]

FORBIDDEN_FEATURE_COLUMNS = set(KEY_COLUMNS + TARGET_COLUMNS + ["origin_season"]) - {
    "AvdelingID",
    "Analyse_Kategori",
}


def feature_set_columns(model_label: str) -> list[str]:
    flag = {
        "M1": "allowed_m1_baseline",
        "M2": "allowed_m2_synthetic_point_weather",
        "M3": "allowed_m3_perfect_information",
        "M4": "allowed_m4_synthetic_engineered_weather",
    }[model_label]
    registry_features = feature_registry.loc[feature_registry[flag], "column"].tolist()
    registry_features = [
        c for c in registry_features if c not in FORBIDDEN_FEATURE_COLUMNS
    ]
    explicit_categoricals = [
        c for c in CATEGORICAL_FEATURE_CANDIDATES if c in {"AvdelingID", "Analyse_Kategori"}
    ]
    return list(dict.fromkeys(explicit_categoricals + registry_features))


# Build all four feature sets so leakage assertions cover M3;
# tuning uses TUNING_FEATURE_SETS.
ALL_LABELS = ["M1", "M2", "M3", "M4"]
feature_sets = {label: feature_set_columns(label) for label in ALL_LABELS}

for label, cols in feature_sets.items():
    print(f"{label}: {len(cols)} features")

leakage_columns = set(feature_registry.loc[feature_registry["leakage_risk"], "column"])
realised_weather_columns = set(
    feature_registry.loc[
        feature_registry["feature_group"] == "realised_weather_perfect_information",
        "column",
    ]
)


def assert_no_leakage(label: str) -> None:
    cols = set(feature_sets[label])
    if label in {"M1", "M2", "M4"}:
        bad = cols & leakage_columns
        if bad:
            raise AssertionError(f"{label} contains leakage-risk columns: {sorted(bad)}")
    elif label == "M3":
        bad = (cols & leakage_columns) - realised_weather_columns
        if bad:
            raise AssertionError(
                f"M3 contains non-realised-weather leakage-risk columns: {sorted(bad)}"
            )
    target_overlap = cols & set(TARGET_COLUMNS)
    if target_overlap:
        raise AssertionError(f"{label} overlaps target columns: {sorted(target_overlap)}")
    key_overlap = cols & {"target_date", "origin_date", "horizon"}
    if key_overlap:
        raise AssertionError(f"{label} overlaps non-categorical key columns: {sorted(key_overlap)}")
    if "origin_season" in cols:
        raise AssertionError(
            f"{label} contains origin_season; must be diagnostic only."
        )


for label in ALL_LABELS:
    assert_no_leakage(label)
print("Feature-set leakage assertions passed for M1, M2, M3, M4.")

# Split selected features into categorical and numeric pipeline inputs.
def split_numeric_categorical(columns: list[str]) -> tuple[list[str], list[str]]:
    categorical = [c for c in columns if c in CATEGORICAL_FEATURE_CANDIDATES]
    numeric = [c for c in columns if c not in CATEGORICAL_FEATURE_CANDIDATES]
    return numeric, categorical


feature_set_numeric: dict[str, list[str]] = {}
feature_set_categorical: dict[str, list[str]] = {}
for label, cols in feature_sets.items():
    num, cat = split_numeric_categorical(cols)
    feature_set_numeric[label] = num
    feature_set_categorical[label] = cat


def assert_m1_m2_m4_have_no_disallowed_columns() -> None:
    """Extra defence: M1/M2/M4 must not contain any pressure/ERA5/climatology/
    apparent-temperature/realised-truth column even if the registry flags would
    allow it. This catches accidental future registry edits.
    """
    disallowed_substrings = (
        "_realised", "_observed", "_obs_", "_error", "_truth",
        "msl", "pressure", "apparent",
        "era5_", "_era5", "clim_", "_climatology",
    )
    realised_truth_cols = set(realised_weather_columns) | set(
        feature_registry.loc[
            feature_registry["feature_group"].isin(
                {"realised_weather_perfect_information"}
            ),
            "column",
        ]
    )
    for label in ["M1", "M2", "M4"]:
        cols = feature_sets[label]
        bad_substring = [c for c in cols if any(s in c.lower() for s in disallowed_substrings)]
        if bad_substring:
            raise AssertionError(
                f"{label} contains disallowed columns by name pattern: {bad_substring}"
            )
        bad_truth = [c for c in cols if c in realised_truth_cols]
        if bad_truth:
            raise AssertionError(
                f"{label} contains realised-weather perfect-information columns: {bad_truth}"
            )


assert_m1_m2_m4_have_no_disallowed_columns()
print("M1/M2/M4 substring + realised-truth assertions passed.")

# M3 is tuned for comparability but remains a non-operational realised-weather benchmark.
if "M3" in TUNING_FEATURE_SETS:
    print(
        "Note: M3 is included in the tuning loop for fair LightGBM comparison. "
        "M3 remains a non-operational realised-weather point-information "
        "benchmark and is not eligible as an operational model in the thesis."
    )

print(f"Tuning feature sets: {TUNING_FEATURE_SETS}")


## Load panel and apply time-safe split

Only the columns required for tuning are loaded: selected feature columns, key columns, and the target. The split uses target dates for the train/validation boundary and an additional origin-date guard for validation rows. After the split, the reserved rolling-origin window is excluded from the in-memory tuning data.


In [ ]:
selected_columns = sorted(
    set(
        KEY_COLUMNS
        + ["Antall_total"]
        + [c for label in ALL_LABELS for c in feature_sets[label]]
    )
)

panel = pd.read_parquet(ML_PANEL_PATH, columns=selected_columns)
panel["target_date"] = pd.to_datetime(panel["target_date"]).dt.normalize()
panel["origin_date"] = pd.to_datetime(panel["origin_date"]).dt.normalize()
panel["horizon"] = panel["horizon"].astype("int8")

panel = panel.loc[panel["horizon"].isin(HORIZONS_TO_TUNE)].reset_index(drop=True)
panel = panel.loc[panel["Antall_total"].notna()].reset_index(drop=True)

# Validation rows must lie before the reserved rolling-origin window.
train_mask = panel["target_date"] <= TUNING_TRAIN_END
val_mask = (
    (panel["target_date"] >= TUNING_VAL_START)
    & (panel["target_date"] <= TUNING_VAL_END)
    & (panel["origin_date"] <= TUNING_VAL_END)
)

train_panel = panel.loc[train_mask].reset_index(drop=True)
val_panel = panel.loc[val_mask].reset_index(drop=True)

# Assert that reserved final-evaluation rows do not enter tuning.
def _assert_reserved_not_in(df: pd.DataFrame, name: str) -> None:
    if df.empty:
        return
    if (df["origin_date"] >= RESERVED_ROLLING_ORIGIN_START).any():
        raise AssertionError(
            f"{name} contains rows whose origin_date is in the reserved "
            f"rolling-origin window (origin_date >= {RESERVED_ROLLING_ORIGIN_START.date()})."
        )


_assert_reserved_not_in(train_panel, "train_panel")
_assert_reserved_not_in(val_panel, "val_panel")

print(f"Total panel rows after horizon filter: {len(panel):,}")
print(f"Tuning train rows:      {len(train_panel):,}")
print(f"Tuning validation rows: {len(val_panel):,}")
print(
    f"Train target_date range: {train_panel['target_date'].min().date()} .. "
    f"{train_panel['target_date'].max().date()}"
)
print(
    f"Val   target_date range: {val_panel['target_date'].min().date()} .. "
    f"{val_panel['target_date'].max().date()}"
)
print(
    f"Val   origin_date range: {val_panel['origin_date'].min().date()} .. "
    f"{val_panel['origin_date'].max().date()}"
)
print(f"Memory (approx): {panel.memory_usage(deep=True).sum() / 1_000_000_000:.3f} GB")

del panel
gc.collect()


## Preprocessing pipeline and metrics

The preprocessing pipeline follows notebook 07: numeric columns pass through, categorical columns are one-hot encoded with `handle_unknown="ignore"`, and LightGBM receives sparse input when appropriate. RMSE is the Optuna objective; MAE, WAPE, and bias are recorded as supporting validation metrics.


In [ ]:
def build_pipeline(numeric_columns: list[str], categorical_columns: list[str], params: dict) -> Pipeline:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    transformer = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_columns),
            ("cat", encoder, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    estimator = LGBMRegressor(
        objective="regression",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbose=-1,
        **params,
    )
    return Pipeline([("preprocess", transformer), ("model", estimator)])


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.mean(np.abs(y_true - y_pred)))


def wape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    denom = float(np.sum(np.abs(y_true)))
    if denom <= 0:
        return float("nan")
    return float(np.sum(np.abs(y_true - y_pred)) / denom)


def bias(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.mean(y_pred - y_true))


def all_metrics(y_true, y_pred) -> dict:
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": mae(y_true, y_pred),
        "wape": wape(y_true, y_pred),
        "bias": bias(y_true, y_pred),
        "n": int(len(y_true)),
    }


## Optuna objective

Each Optuna trial fits one LightGBM pipeline on the tuning train set, predicts on the tuning validation set, clips negative predictions to zero, and returns pooled validation RMSE on the original sales scale across `HORIZONS_TO_TUNE`.

| Parameter | Range | Sampling | Notes |
| --- | --- | --- | --- |
| `n_estimators` | 200 .. 1500 | int | Higher upper bound than in 07 to give Optuna room. |
| `learning_rate` | 0.01 .. 0.20 | log-uniform | |
| `num_leaves` | 16 .. 255 | int | LightGBM main capacity knob. |
| `max_depth` | -1, 4 .. 12 | mixed | -1 = unlimited; explicit caps for regularisation. |
| `min_child_samples` | 10 .. 200 | int | Leaf-size regularisation. |
| `subsample` | 0.6 .. 1.0 | uniform | Bagging fraction. |
| `subsample_freq` | 0 .. 5 | int | Bagging frequency. |
| `colsample_bytree` | 0.6 .. 1.0 | uniform | Feature subsampling. |
| `reg_alpha` | 1e-8 .. 5.0 | log-uniform | L1 regularisation. |
| `reg_lambda` | 1e-8 .. 5.0 | log-uniform | L2 regularisation. |
| `min_split_gain` | 0.0 .. 1.0 | uniform | Minimum gain to split a node. |

The search ranges avoid extreme values that have produced unstable LightGBM training.


In [ ]:
def suggest_params(trial: "optuna.Trial") -> dict:
    # Mixed max_depth sampling: -1 for unlimited depth or explicit regularising caps.
    use_max_depth = trial.suggest_categorical("use_max_depth", [False, True])
    if use_max_depth:
        max_depth = trial.suggest_int("max_depth", 4, 12)
    else:
        max_depth = -1

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 255),
        "max_depth": max_depth,
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
    }
    return params


def fit_predict_clip(numeric_cols, categorical_cols, params, X_train, y_train, X_val):
    pipeline = build_pipeline(numeric_cols, categorical_cols, params)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline.fit(X_train, y_train)
    y_pred = np.asarray(pipeline.predict(X_val), dtype="float64")
    if CLIP_NEGATIVE_PREDICTIONS:
        y_pred = np.clip(y_pred, 0.0, None)
    return pipeline, y_pred


def build_objective(label: str):
    numeric_cols = feature_set_numeric[label]
    categorical_cols = feature_set_categorical[label]
    feature_cols = numeric_cols + categorical_cols
    X_train = train_panel[feature_cols]
    y_train = train_panel["Antall_total"].astype("float32").to_numpy()
    X_val = val_panel[feature_cols]
    y_val = val_panel["Antall_total"].astype("float32").to_numpy()

    def objective(trial):
        params = suggest_params(trial)
        try:
            _, y_pred = fit_predict_clip(
                numeric_cols, categorical_cols, params, X_train, y_train, X_val
            )
        except Exception as exc:
            trial.set_user_attr("error", f"{type(exc).__name__}: {exc}")
            return float("inf")
        m = all_metrics(y_val, y_pred)
        trial.set_user_attr("rmse", m["rmse"])
        trial.set_user_attr("mae", m["mae"])
        trial.set_user_attr("wape", m["wape"])
        trial.set_user_attr("bias", m["bias"])
        trial.set_user_attr("n", m["n"])
        return m["rmse"]

    return objective


## Tuning execution

For each feature set in `TUNING_FEATURE_SETS`, the notebook runs an Optuna study with `N_TRIALS` trials. Studies use a fixed TPE sampler seed for reproducibility, and median pruning is enabled only in full mode.


In [ ]:
sampler = TPESampler(seed=OPTUNA_SAMPLER_SEED)
pruner = MedianPruner(n_warmup_steps=5) if ENABLE_OPTUNA_PRUNING else NopPruner()

studies: dict[str, "optuna.Study"] = {}
study_runtimes: dict[str, float] = {}

for label in TUNING_FEATURE_SETS:
    print(f"\n=== Tuning {label} ({N_TRIALS} trials) ===")
    study = optuna.create_study(
        direction=OPTUNA_DIRECTION,
        sampler=sampler,
        pruner=pruner,
        study_name=f"lightgbm_{label}_{RUN_MODE}",
    )
    t0 = time.time()
    study.optimize(build_objective(label), n_trials=N_TRIALS, show_progress_bar=False)
    elapsed = time.time() - t0
    studies[label] = study
    study_runtimes[label] = elapsed
    best = study.best_trial
    print(f"  best RMSE: {best.value:.4f}")
    print(f"  best params: {best.params}")
    print(f"  elapsed: {elapsed/60:.2f} min")


## Validation metrics for selected parameters

After tuning, the notebook refits one LightGBM model per feature set with the selected Optuna parameters and computes pooled and per-horizon validation metrics. These validation tables document the tuned configurations before final rolling-origin evaluation.


In [ ]:
def _resolve_best_params(study) -> dict:
    """Translate the Optuna param dict (which contains use_max_depth) into the
    LightGBM keyword arguments the pipeline builder expects."""
    raw = dict(study.best_params)
    use_max_depth = bool(raw.pop("use_max_depth", False))
    if not use_max_depth:
        # If max_depth was not sampled, store -1 for unlimited depth.
        raw["max_depth"] = -1
    return raw


validation_rows = []
best_params_per_label: dict[str, dict] = {}

for label, study in studies.items():
    best_params = _resolve_best_params(study)
    best_params_per_label[label] = best_params

    numeric_cols = feature_set_numeric[label]
    categorical_cols = feature_set_categorical[label]
    feature_cols = numeric_cols + categorical_cols
    X_train = train_panel[feature_cols]
    y_train = train_panel["Antall_total"].astype("float32").to_numpy()
    X_val = val_panel[feature_cols]
    y_val = val_panel["Antall_total"].astype("float32").to_numpy()

    _, y_pred = fit_predict_clip(
        numeric_cols, categorical_cols, best_params, X_train, y_train, X_val
    )

    pooled_metrics = all_metrics(y_val, y_pred)
    validation_rows.append(
        {
            "run_mode": RUN_MODE,
            "feature_set": label,
            "scope": "pooled_all_horizons",
            "horizon": -1,
            **pooled_metrics,
        }
    )

    # Per-horizon validation diagnostics for the evaluation notebook.
    horizons_in_val = val_panel["horizon"].to_numpy()
    for h in sorted(np.unique(horizons_in_val).tolist()):
        mask = horizons_in_val == h
        if mask.sum() == 0:
            continue
        m = all_metrics(y_val[mask], y_pred[mask])
        validation_rows.append(
            {
                "run_mode": RUN_MODE,
                "feature_set": label,
                "scope": "by_horizon",
                "horizon": int(h),
                **m,
            }
        )

validation_metrics_df = pd.DataFrame(validation_rows)
print("Validation metrics (pooled) per feature set:")
display(
    validation_metrics_df.loc[validation_metrics_df["scope"] == "pooled_all_horizons"]
    .set_index("feature_set")[["rmse", "mae", "wape", "bias", "n"]].round(4)
)


## Save parameters and diagnostics

The notebook writes the selected parameter payload, trial log, validation metrics, feature-set summary, and runtime summary:

- `lightgbm_optuna_best_params_{run_mode}.csv`
- `lightgbm_optuna_best_params_{run_mode}.json`
- `lightgbm_optuna_trials_{run_mode}.csv`
- `lightgbm_optuna_validation_metrics_{run_mode}.csv`
- `lightgbm_optuna_feature_set_summary_{run_mode}.csv`
- `lightgbm_optuna_runtime_summary_{run_mode}.csv`


In [ ]:
best_params_rows = []
best_params_payload: dict = {
    "run_mode": RUN_MODE,
    "n_trials": N_TRIALS,
    "tuning_train_end": str(TUNING_TRAIN_END.date()),
    "tuning_val_start": str(TUNING_VAL_START.date()),
    "tuning_val_end": str(TUNING_VAL_END.date()),
    "reserved_rolling_origin_start": str(RESERVED_ROLLING_ORIGIN_START.date()),
    "reserved_rolling_origin_end": str(RESERVED_ROLLING_ORIGIN_END.date()),
    "horizons_tuned": HORIZONS_TO_TUNE,
    "tuning_feature_sets": TUNING_FEATURE_SETS,
    "lightgbm_static_kwargs": {
        "objective": "regression",
        "n_jobs": N_JOBS,
        "random_state": RANDOM_STATE,
        "verbose": -1,
    },
    "feature_sets": {},
}
for label, study in studies.items():
    best_params = best_params_per_label[label]
    row = {
        "run_mode": RUN_MODE,
        "feature_set": label,
        "best_value_rmse": float(study.best_value),
        "n_trials": N_TRIALS,
        "n_features": len(feature_sets[label]),
        "n_numeric": len(feature_set_numeric[label]),
        "n_categorical": len(feature_set_categorical[label]),
    }
    row.update({f"best_param_{k}": v for k, v in best_params.items()})
    best_params_rows.append(row)
    best_params_payload["feature_sets"][label] = {
        "best_value_rmse": float(study.best_value),
        "best_params": best_params,
        "numeric_columns": feature_set_numeric[label],
        "categorical_columns": feature_set_categorical[label],
    }

best_params_df = pd.DataFrame(best_params_rows)
best_params_df.to_csv(BEST_PARAMS_CSV, index=False)
BEST_PARAMS_JSON.write_text(json.dumps(best_params_payload, indent=2, default=str), encoding="utf-8")
print(f"Saved: {project_relative(BEST_PARAMS_CSV)}")
print(f"Saved: {project_relative(BEST_PARAMS_JSON)}")

trial_rows = []
for label, study in studies.items():
    for trial in study.trials:
        trial_rows.append({
            "run_mode": RUN_MODE,
            "feature_set": label,
            "trial_number": trial.number,
            "state": str(trial.state).split(".")[-1],
            "value": trial.value if trial.value is not None else float("nan"),
            "datetime_start": trial.datetime_start.isoformat() if trial.datetime_start else "",
            "datetime_complete": trial.datetime_complete.isoformat() if trial.datetime_complete else "",
            "duration_seconds": trial.duration.total_seconds() if trial.duration else float("nan"),
            "user_attr_rmse": trial.user_attrs.get("rmse", float("nan")),
            "user_attr_mae": trial.user_attrs.get("mae", float("nan")),
            "user_attr_wape": trial.user_attrs.get("wape", float("nan")),
            "user_attr_bias":  trial.user_attrs.get("bias", float("nan")),
            "user_attr_n":     trial.user_attrs.get("n", float("nan")),
            "user_attr_error": trial.user_attrs.get("error", ""),
            **{f"param_{k}": v for k, v in trial.params.items()},
        })
trials_df = pd.DataFrame(trial_rows)
trials_df.to_csv(TRIALS_CSV, index=False)
print(f"Saved: {project_relative(TRIALS_CSV)} ({len(trials_df):,} rows)")

validation_metrics_df.to_csv(VAL_METRICS_CSV, index=False)
print(f"Saved: {project_relative(VAL_METRICS_CSV)} ({len(validation_metrics_df):,} rows)")

fss_rows = []
for label in TUNING_FEATURE_SETS:
    cols = feature_sets[label]
    fss_rows.append({
        "run_mode": RUN_MODE,
        "feature_set": label,
        "n_features": len(cols),
        "n_numeric":  len(feature_set_numeric[label]),
        "n_categorical": len(feature_set_categorical[label]),
        "feature_columns": ";".join(cols),
    })
feature_set_summary_df = pd.DataFrame(fss_rows)
feature_set_summary_df.to_csv(FEATURE_SET_CSV, index=False)
print(f"Saved: {project_relative(FEATURE_SET_CSV)}")

runtime_df = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        "feature_set": label,
        "n_trials": N_TRIALS,
        "elapsed_seconds": float(study_runtimes[label]),
        "elapsed_minutes": float(study_runtimes[label]) / 60.0,
    }
    for label in TUNING_FEATURE_SETS
])
runtime_df.to_csv(RUNTIME_CSV, index=False)
print(f"Saved: {project_relative(RUNTIME_CSV)}")


## Optional plots

If matplotlib is available, the notebook writes diagnostic figures to `outputs/lightgbm_tuning/figures_{run_mode}/`: `optimization_history_{label}.png`, `parameter_importance_{label}.png`, and `validation_rmse_by_feature_set.png`. CSV and JSON files remain the canonical tuning outputs.


In [ ]:
if _MPL_OK and studies:
    for label, study in studies.items():
        vals = [t.value for t in study.trials if t.value is not None and np.isfinite(t.value)]
        if vals:
            fig, ax = plt.subplots(figsize=(7.5, 4.5))
            ax.plot(range(1, len(vals) + 1), vals, marker="o")
            best_so_far = np.minimum.accumulate(vals)
            ax.plot(range(1, len(vals) + 1), best_so_far, linestyle="--", label="best so far")
            ax.set_xlabel("Trial number")
            ax.set_ylabel("Validation RMSE")
            ax.set_title(f"Optuna optimization history - LightGBM {label} ({RUN_MODE} mode)")
            ax.legend()
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            fig.savefig(FIG_DIR / f"optimization_history_{label}.png", dpi=140)
            plt.close(fig)
            print(f"Saved: figures_{RUN_MODE}/optimization_history_{label}.png")

        try:
            importance = optuna.importance.get_param_importances(study)
            if importance:
                names = list(importance.keys())
                values = list(importance.values())
                fig, ax = plt.subplots(figsize=(7.5, 0.35 * max(len(names), 4) + 1.5))
                ax.barh(names[::-1], values[::-1])
                ax.set_xlabel("Importance")
                ax.set_title(f"Optuna parameter importance - LightGBM {label} ({RUN_MODE} mode)")
                ax.grid(True, axis="x", alpha=0.3)
                fig.tight_layout()
                fig.savefig(FIG_DIR / f"parameter_importance_{label}.png", dpi=140)
                plt.close(fig)
                print(f"Saved: figures_{RUN_MODE}/parameter_importance_{label}.png")
        except Exception as exc:
            print(f"Skipped parameter importance for {label}: {type(exc).__name__}: {exc}")

    # Cross-feature-set validation RMSE diagnostic.
    pooled = validation_metrics_df.loc[validation_metrics_df["scope"] == "pooled_all_horizons"]
    if not pooled.empty:
        fig, ax = plt.subplots(figsize=(6.5, 4.0))
        ax.bar(pooled["feature_set"], pooled["rmse"])
        ax.set_xlabel("Feature set")
        ax.set_ylabel("Pooled validation RMSE")
        ax.set_title(f"LightGBM tuned validation RMSE ({RUN_MODE} mode)")
        for x, y in zip(pooled["feature_set"], pooled["rmse"]):
            ax.text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=9)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "validation_rmse_by_feature_set.png", dpi=140)
        plt.close(fig)
        print(f"Saved: figures_{RUN_MODE}/validation_rmse_by_feature_set.png")
else:
    if not _MPL_OK:
        print("matplotlib unavailable; skipping plots.")


## Notes for rolling-origin evaluation

The rolling-origin evaluation notebook should read `outputs/lightgbm_tuning/lightgbm_optuna_best_params_full.json` and instantiate `LGBMRegressor` per feature set. The intended evaluation uses forecast origins in the reserved period, trains on rows available at each origin, predicts horizons `h = 0..10`, and reports metrics by horizon and horizon tier. Overlapping target dates are allowed because they represent distinct forecasts made at different lead times.

The separation between tuning and final evaluation prevents tuning-period leakage into thesis-reported results. M3 must remain labelled as a non-operational realised-weather point-information benchmark.


## Summary

This notebook tunes LightGBM with Optuna for M1, M2, M3, and M4 using a forecast-origin-safe train/validation split before the reserved rolling-origin period. M3 is included for a fair LightGBM comparison but remains non-operational.

The canonical full-run outputs are `outputs/lightgbm_tuning/lightgbm_optuna_best_params_full.json`, `outputs/lightgbm_tuning/lightgbm_optuna_best_params_full.csv`, `outputs/lightgbm_tuning/lightgbm_optuna_trials_full.csv`, `outputs/lightgbm_tuning/lightgbm_optuna_validation_metrics_full.csv`, `outputs/lightgbm_tuning/lightgbm_optuna_feature_set_summary_full.csv`, and `outputs/lightgbm_tuning/lightgbm_optuna_runtime_summary_full.csv`. Optional full-run plots are written under `outputs/lightgbm_tuning/figures_full/`.
